# **Question 7: Advanced - Functional Programming & Pipelines**

In MLOps data processing, we often chain operations: `load -> clean -> normalize -> validate`.

Python has functional tools like `map`, `filter`, and `reduce`.

**The Scenario:**
You have a list of probability scores: `scores = [0.1, 0.4, 0.8, 0.9, 0.3, 0.6]`.
You need to:
1.  **Filter** out scores below 0.5.
2.  **Map** (convert) the remaining scores to percentage strings (e.g., "80%").

**The Question:**
1.  Write a **one-liner** using `map` and `filter` (and `lambda` functions) to achieve this.
2.  **The MLOps Reality Check:** While `map/filter` are "functional," standard Python List Comprehensions are often preferred in Pythonic code. **Why?** (Hint: It's about Readability).
3.  How would the library `itertools` help if this `scores` list was actually an **infinite stream** of data coming from a Kafka topic?

### Part 1: One-Liner Solution

```python
result = list(map(lambda x: f"{x:.0%}", filter(lambda x: x >= 0.5, scores)))
# Output: ['80%', '90%', '60%']
```

**Step-by-step breakdown:**
1. `filter(lambda x: x >= 0.5, scores)` → Keeps `[0.8, 0.9, 0.6]`
2. `map(lambda x: f"{x:.0%}", ...)` → Converts to `['80%', '90%', '60%']`
3. `list(...)` → Materializes the iterator into a list

**Key formatting note:** The `:.0%` format specifier automatically multiplies by 100 and adds the `%` symbol.

---

### Part 2: Why List Comprehensions Are Preferred

**The Pythonic alternative:**
```python
result = [f"{x:.0%}" for x in scores if x >= 0.5]
```

**Reasons for preference:**

| Aspect | `map`/`filter` | List Comprehension |
|--------|----------------|-------------------|
| **Readability** | Nested, inside-out logic | Linear, left-to-right flow |
| **Syntax** | Requires `lambda` or function | Native Python syntax |
| **Debugging** | Harder to inspect intermediate steps | Easier to print/debug |
| **Performance** | Slightly slower (function calls) | Marginally faster in CPython |
| **PEP 8** | Discouraged for simple cases | Explicitly recommended |

**Interview talking point:**
> "While `map`/`filter` are functional programming concepts, Python isn't purely functional. List comprehensions are more **Pythonic** because they prioritize **readability** and **maintainability** over functional purity. In production MLOps code, readable code is debuggable code."

---

### Part 3: Infinite Streams with `itertools`

**What is Apache Kafka?**
- Distributed event-streaming platform for real-time data pipelines
- Publishes, stores, and processes data streams with high throughput
- Common in MLOps for real-time feature engineering and model inference

**The Challenge:**
When consuming from Kafka, you can't load "all data" into memory—it's an **infinite stream**.

**How `itertools` Solves This:**

```python
from itertools import islice, takewhile, filterfalse

# Simulated Kafka stream (infinite iterator)
def kafka_stream():
    """Infinite stream of probability scores"""
    from itertools import cycle, count
    import random
    for _ in count():
        yield random.random()

# Process stream lazily (no memory explosion)
stream = kafka_stream()

# Example 1: Process in batches of 100
from itertools import islice
def process_batch(stream, batch_size=100):
    while True:
        batch = list(islice(stream, batch_size))
        if not batch:
            break
        # Process batch
        valid_scores = [f"{x:.0%}" for x in batch if x >= 0.5]
        yield valid_scores

# Example 2: Take while condition holds
from itertools import takewhile
high_scores = takewhile(lambda x: x >= 0.5, stream)

# Example 3: Filter and format lazily
filtered = filter(lambda x: x >= 0.5, stream)
formatted = map(lambda x: f"{x:.0%}", filtered)
# Still lazy! Only computes when you iterate
```

---

## 🧠 Core Concepts

### 1. Lazy Evaluation

**Why it matters:**
- **Memory efficient:** Iterators don't store all results
- **Stream-friendly:** Works with infinite data sources
- **Pipeline composability:** Chain operations without intermediate lists

```python
# Eager (loads everything)
result = [f"{x:.0%}" for x in scores if x >= 0.5]  # Full list in memory

# Lazy (computes on-demand)
result = map(lambda x: f"{x:.0%}", filter(lambda x: x >= 0.5, scores))
# Nothing computed yet! Only when you iterate or call list()
```

### 2. Iterator Mental Model

```
map/filter/itertools → Iterator (lazy, one-pass, memory-efficient)
List comprehension    → List (eager, reusable, memory-heavy)
```

**When to use each:**
- **Iterators:** Streaming data, large datasets, pipelines
- **Lists:** Small data, need to reuse, need indexing

### 3. Key Functional Operations

| Function | Purpose | Example |
|----------|---------|---------|
| `map` | Transform each element | `map(lambda x: x*2, [1,2,3])` → `[2,4,6]` |
| `filter` | Keep elements matching condition | `filter(lambda x: x>2, [1,2,3])` → `[3]` |
| `reduce` | Aggregate to single value | `reduce(lambda a,b: a+b, [1,2,3])` → `6` |

---

## 🔹 Bonus: Essential `itertools` for MLOps

### Common Patterns

```python
from itertools import islice, chain, cycle, groupby, accumulate

# 1. Batch processing (chunking)
def chunked(iterable, size):
    """Split data into fixed-size chunks"""
    it = iter(iterable)
    while chunk := list(islice(it, size)):
        yield chunk

# Usage: for batch in chunked(huge_dataset, 1000): ...

# 2. Sliding window (time-series features)
def sliding_window(iterable, size):
    """Rolling window for feature engineering"""
    it = iter(iterable)
    window = list(islice(it, size))
    if len(window) == size:
        yield tuple(window)
    for item in it:
        window = window[1:] + [item]
        yield tuple(window)

# 3. Infinite cycle (data augmentation)
from itertools import cycle
train_data = [batch1, batch2, batch3]
infinite_train = cycle(train_data)  # Repeats forever

# 4. Chain multiple iterators (combine datasets)
from itertools import chain
all_data = chain(train_set, val_set, test_set)

# 5. Running statistics (cumulative metrics)
from itertools import accumulate
losses = [0.5, 0.4, 0.3, 0.2]
cumulative_loss = list(accumulate(losses))  # [0.5, 0.9, 1.2, 1.4]
```

### Why `itertools` for Kafka/Streaming

**Problem:** Kafka topics are infinite—you can't `list(kafka_consumer)`.

**Solution:** Process lazily with `itertools`:

```python
from kafka import KafkaConsumer
from itertools import islice

consumer = KafkaConsumer('ml-predictions')

# Process in micro-batches
for batch in chunked(consumer, 100):
    # Extract scores
    scores = [msg.value['score'] for msg in batch]
    # Apply pipeline
    valid = filter(lambda x: x >= 0.5, scores)
    formatted = map(lambda x: f"{x:.0%}", valid)
    # Materialize only what you need
    results = list(formatted)
    # Send to downstream system
    process(results)
```

**Key advantages:**
- ✅ Constant memory usage (O(batch_size), not O(stream_size))
- ✅ Real-time processing (no need to wait for "all data")
- ✅ Fault tolerance (process chunk-by-chunk with checkpoints)

---

## 🎯 Interview Talking Points

### Strong Statements to Make:

1. **On functional vs. Pythonic:**
   > "While `map`/`filter` are valid, I prefer list comprehensions for simple transformations because they're more readable. In production code, especially in collaborative ML teams, **readability is a feature, not a trade-off.**"

2. **On lazy evaluation:**
   > "For large-scale data pipelines, lazy evaluation with iterators prevents memory exhaustion. In MLOps, we often process gigabytes of data—eager evaluation would crash the system."

3. **On streaming:**
   > "When consuming from Kafka or other streaming sources, `itertools` lets me build memory-safe pipelines. I can process infinite streams in fixed-memory batches, which is critical for production ML systems."

4. **On choosing the right tool:**
   > "I use list comprehensions for small, finite datasets where readability matters. I switch to `itertools` when dealing with streams, large files, or when I need to chain multiple lazy operations efficiently."

---

## 📚 Quick Reference

### One-Line Mental Models

```
map      → "Change each item"
filter   → "Keep some items"
reduce   → "Combine all items"

itertools → "Lego blocks for lazy loops"
```

### When to Use What

| Scenario | Tool | Reason |
|----------|------|--------|
| Small list transformation | List comprehension | Readability |
| Large file processing | `itertools` + generators | Memory |
| Kafka/streaming | `itertools` | Infinite streams |
| Complex pipeline | Generator functions | Control flow |
| Simple aggregation | `reduce` or `sum()` | Clarity |

---

## ⚠️ Common Pitfalls

### 1. Logic Inversion
```python
# ❌ WRONG: Keeps scores BELOW 0.5
filter(lambda x: x < 0.5, scores)

# ✅ CORRECT: Keeps scores ABOVE OR EQUAL to 0.5
filter(lambda x: x >= 0.5, scores)
```

### 2. Format String Confusion
```python
# ❌ Wrong: Returns 80.0 (not a percentage string)
map(lambda x: x * 100, scores)

# ✅ Correct: Returns "80%" (formatted string)
map(lambda x: f"{x:.0%}", scores)
```

### 3. Iterator Exhaustion
```python
# ❌ Can't reuse iterators
result = map(lambda x: x*2, scores)
list(result)  # [0.2, 0.8, 1.6, 1.8, 0.6, 1.2]
list(result)  # [] ← Iterator exhausted!

# ✅ Create new iterator or use list
result = [x*2 for x in scores]  # Reusable
```

---

## 🔗 Related Topics for Follow-Up

- Generator functions and `yield`
- Memory profiling with `memory_profiler`
- Apache Kafka consumer patterns
- Data pipeline frameworks (Apache Beam, Prefect)
- Python's `functools` module (`partial`, `lru_cache`)

---

*Last updated: For MLOps/ML Engineer interview preparation*